# SPIDER ML Task 2 - Applied ML
## Trustworthy Healthcare Information Assistant

In this notebook, I will build a simple medical question-answering assistant using retrieval.
The assistant will search relevant medical evidence, generate grounded responses, estimate confidence, and handle unsafe queries carefully.


## Problem Statement

The goal is to build a healthcare assistant that does not simply chat freely.
Instead, it should retrieve relevant medical evidence from a trusted dataset, use that evidence to answer questions, show citations, and avoid unsafe advice.

In [1]:
!pip -q install sentence-transformers faiss-cpu scikit-learn pandas numpy tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 38.8 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import faiss
from sentence_transformers import SentenceTransformer

## Load Dataset

We will use a medical question-answer dataset as the knowledge source for the assistant.
Each row should contain a question and an answer that can be used as retrieval evidence.

In [4]:
df = pd.read_csv("medquad.csv", engine="python", on_bad_lines="skip")
df.head()

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


In [5]:
print(df.columns)
print(df.shape)

Index(['question', 'answer', 'source', 'focus_area'], dtype='object')
(8354, 4)


## Data Cleaning

We remove empty rows and keep only the columns needed for the assistant.

In [6]:
df = df.dropna().copy()
df.head()

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


## Normalize Columns

If the dataset uses different column names, we will map them to a common format.

In [7]:
df.columns = [c.lower().strip() for c in df.columns]
print(df.columns)

Index(['question', 'answer', 'source', 'focus_area'], dtype='object')


## Ready for Retrieval

Now the dataset is prepared for building a searchable medical knowledge base.

## Build Medical Knowledge Base

We convert each question-answer pair into a document.
Then we create embeddings and store them in a FAISS index for fast semantic search.

In [8]:
# Make sure the dataset has question and answer columns
print(df.columns)
print(df.shape)

# Keep only the important columns if they exist
# Change these names if your dataset uses different ones
df = df.rename(columns={"question": "question", "answer": "answer"})

# Remove empty values and clean text
df = df.dropna(subset=["question", "answer"]).copy()
df["question"] = df["question"].astype(str).str.strip()
df["answer"] = df["answer"].astype(str).str.strip()



Index(['question', 'answer', 'source', 'focus_area'], dtype='object')
(8335, 4)


In [9]:
# Create document list for retrieval
documents = []
for i, row in df.iterrows():
    documents.append({
        "id": i,
        "question": row["question"],
        "answer": row["answer"],
        "text": f"Question: {row['question']}\nAnswer: {row['answer']}"
    })

print("Total documents:", len(documents))
documents[0]

Total documents: 8335


{'id': 0,
 'question': 'What is (are) Glaucoma ?',
 'answer': "Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops  There are several different types of glaucoma. Most of these involve the drainage system within the eye. At the front of the eye there is a small space called the anterior chamber. A clear fluid flows through this chamber and bathes and nourishes the nearby tissues. (Watch the video to learn more about glaucoma. To enlarge the video, click the brackets in the lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.) In glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. As the fluid builds up, the pressure inside the eye rises. Unless this pressure is controlled, it may cause damage to the optic nerve and other parts of the eye and result in loss of 

## Text Embeddings

We use SentenceTransformer to convert each document into a vector.
Similar questions will have similar vectors.

In [10]:
# Load the embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Extract document texts
doc_texts = [doc["text"] for doc in documents]

# Create embeddings
doc_embeddings = embedder.encode(
    doc_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", doc_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/261 [00:00<?, ?it/s]

Embedding shape: (8335, 384)


## FAISS Index

We store the document embeddings in FAISS so we can quickly search for the most similar medical evidence.

In [11]:
# Get embedding dimension
dim = doc_embeddings.shape[1]

# Create FAISS index using inner product
index = faiss.IndexFlatIP(dim)

# Add embeddings to the index
index.add(doc_embeddings.astype("float32"))

print("Total vectors in FAISS index:", index.ntotal)

Total vectors in FAISS index: 8335


## Retrieval Function

Given a user query, we will search the FAISS index and return the most relevant documents.

In [12]:
def retrieve(query, top_k=5):
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "question": documents[idx]["question"],
            "answer": documents[idx]["answer"],
            "text": documents[idx]["text"]
        })
    return results

## Test Retrieval

We will check whether the search function returns relevant medical entries.

In [13]:
sample_query = "What are common symptoms of diabetes?"
results = retrieve(sample_query, top_k=3)

for i, r in enumerate(results, 1):
    print(f"\nResult {i}")
    print("Score:", r["score"])
    print("Question:", r["question"])
    print("Answer:", r["answer"][:300])


Result 1
Score: 0.7612723708152771
Question: What are the symptoms of Diabetes ?
Answer: Diabetes is often called a "silent" disease because it can cause serious complications even before you have symptoms. Symptoms can also be so mild that you dont notice them. An estimated 8 million people in the United States have type 2 diabetes and dont know it, according to 2012 estimates by the C

Result 2
Score: 0.7576074600219727
Question: What are the symptoms of Diabetes ?
Answer: Many people with diabetes experience one or more symptoms, including extreme thirst or hunger, a frequent need to urinate and/or fatigue. Some lose weight without trying. Additional signs include sores that heal slowly, dry, itchy skin, loss of feeling or tingling in the feet and blurry eyesight. So

Result 3
Score: 0.6727579832077026
Question: What is (are) Diabetes Type 1 ?
Answer: Diabetes means your blood glucose, or blood sugar, levels are too high. With type 1 diabetes, your pancreas does not make insulin. I

## Grounded Answering

The assistant should answer using only the retrieved evidence.
If the evidence is weak or the request is unsafe, it should refuse or give a safety warning

In [14]:
def build_context(results):
    context_parts = []
    for i, r in enumerate(results, 1):
        context_parts.append(
            f"[Source {i}]\nQuestion: {r['question']}\nAnswer: {r['answer']}"
        )
    return "\n\n".join(context_parts)

def generate_answer(query, results):
    if not results:
        return "I could not find enough relevant medical evidence to answer this safely."

    best = results[0]
    answer = (
        f"Based on the retrieved medical evidence, the most relevant information for your question is:\n\n"
        f"{best['answer']}\n\n"
        f"This answer is supported by the retrieved source related to: {best['question']}"
    )
    return answer

## Confidence Estimation

We estimate confidence from retrieval scores.
A higher average similarity score means the assistant is more confident.

In [15]:
def estimate_confidence(results):
    if not results:
        return 0.0
    scores = [r["score"] for r in results]
    return float(np.clip(np.mean(scores), 0.0, 1.0))

## Safety Layer

Unsafe medical requests should not receive normal answers.
The assistant should refuse emergencies and direct the user to urgent professional help.

In [16]:
unsafe_keywords = [
    "suicide", "kill myself", "self harm", "overdose", "poison",
    "chest pain", "can't breathe", "emergency", "severe bleeding"
]

def is_unsafe(query):
    q = query.lower()
    return any(word in q for word in unsafe_keywords)

def safety_response():
    return (
        "This may be a medical emergency or unsafe situation. "
        "Please contact emergency services or a licensed healthcare professional immediately."
    )

## Assistant Function

This function combines retrieval, grounding, confidence, and safety into one response pipeline.

In [17]:
def ask_assistant(query, top_k=5):
    if is_unsafe(query):
        return {
            "answer": safety_response(),
            "confidence": 0.0,
            "citations": [],
            "retrieved": []
        }

    results = retrieve(query, top_k=top_k)
    confidence = estimate_confidence(results)
    answer_text = generate_answer(query, results)

    citations = []
    for i, r in enumerate(results, 1):
        citations.append(f"[Source {i}] {r['question']}")

    return {
        "answer": answer_text,
        "confidence": confidence,
        "citations": citations,
        "retrieved": results
    }

## Test the Assistant

We test the assistant with a normal medical question and an unsafe query.

In [18]:
test_query_1 = "What are common symptoms of diabetes?"
out1 = ask_assistant(test_query_1)
print(out1["answer"])
print("Confidence:", out1["confidence"])
print("Citations:", out1["citations"])
print("\n")

test_query_2 = "I have chest pain right now"
out2 = ask_assistant(test_query_2)
print(out2["answer"])

Based on the retrieved medical evidence, the most relevant information for your question is:

Diabetes is often called a "silent" disease because it can cause serious complications even before you have symptoms. Symptoms can also be so mild that you dont notice them. An estimated 8 million people in the United States have type 2 diabetes and dont know it, according to 2012 estimates by the Centers for Disease Control and Prevention (CDC). Common Signs Some common symptoms of diabetes are: - being very thirsty  - frequent urination  - feeling very hungry or tired  - losing weight without trying  - having sores that heal slowly  - having dry, itchy skin  - loss of feeling or tingling in the feet  - having blurry eyesight. being very thirsty frequent urination feeling very hungry or tired losing weight without trying having sores that heal slowly having dry, itchy skin loss of feeling or tingling in the feet having blurry eyesight. Signs of type 1 diabetes usually develop over a short per

## Evaluation

We test the assistant on a few sample questions to check retrieval quality, grounded response behavior, confidence, and safety handling.

In [20]:
sample_queries = [
    "What are the symptoms of diabetes?",
    "How can I lower high blood pressure naturally?",
    "I have chest pain right now"
]

for q in sample_queries:
    print("=" * 80)
    print("Question:", q)
    out = ask_assistant(q)
    print("Answer:", out["answer"])
    print("Confidence:", out["confidence"])
    print("Citations:")
    for c in out["citations"]:
        print("-", c)
    print()

Question: What are the symptoms of diabetes?
Answer: Based on the retrieved medical evidence, the most relevant information for your question is:

Many people with diabetes experience one or more symptoms, including extreme thirst or hunger, a frequent need to urinate and/or fatigue. Some lose weight without trying. Additional signs include sores that heal slowly, dry, itchy skin, loss of feeling or tingling in the feet and blurry eyesight. Some people with diabetes, however, have no symptoms at all.

This answer is supported by the retrieved source related to: What are the symptoms of Diabetes ?
Confidence: 0.6881772041320801
Citations:
- [Source 1] What are the symptoms of Diabetes ?
- [Source 2] What are the symptoms of Diabetes ?
- [Source 3] What is (are) Diabetes Type 1 ?
- [Source 4] What is (are) Diabetes Type 2 ?
- [Source 5] What are the symptoms of Maturity-onset diabetes of the young, type 5 ?

Question: How can I lower high blood pressure naturally?
Answer: Based on the re

## Architecture Summary

This assistant follows a simple RAG pipeline:
1. Load medical question-answer data.
2. Convert text into embeddings.
3. Store embeddings in a FAISS index.
4. Retrieve the most relevant evidence for a user question.
5. Build a grounded response from the retrieved evidence.
6. Estimate confidence.
7. Refuse unsafe medical queries.

## Conclusion

In this notebook, I built a simple healthcare information assistant using retrieval-based machine learning.
The system is grounded in retrieved evidence, provides confidence estimates, and includes a safety layer for unsafe requests.